# Exercise: an onion-shaped metadata schema
### Describe your brick model layer by layer, then draw it as a graph

```
Model
 └── Sub-assembly
      └── Piece use
           └── Piece   (shared catalogue)
```

**What you hand in**
1. the tables of your schema (section 1);
2. your JSON files in `sources/` (section 2);
3. your graph and the answers to the questions (sections 4 and 5).

## 0. Setup
Run these two cells first.

In [ ]:
%pip install -q networkx matplotlib jsonschema

In [ ]:
import json, os
import networkx as nx
import matplotlib.pyplot as plt

os.makedirs("sources", exist_ok=True)   # all your files go in ./sources
print("ready")

---
## 1. Design the schema as tables

Double-click this cell and fill in one table per layer. Add as many rows as you need.

**Piece** (in the shared catalogue)

| field | type | required | allowed values | description |
|---|---|---|---|---|
| | | | | |

**Colour** (controlled vocabulary)

| field | type | required | allowed values | description |
|---|---|---|---|---|
| | | | | |

**Piece use** (inside a sub-assembly)

| field | type | required | allowed values | description |
|---|---|---|---|---|
| | | | | |

**Sub-assembly**

| field | type | required | allowed values | description |
|---|---|---|---|---|
| | | | | |

**Model**

| field | type | required | allowed values | description |
|---|---|---|---|---|
| | | | | |

Questions to answer before writing JSON:
- Where does the **colour** go: in the piece or in the piece use? Why?
- How does an outer layer **point** to an inner one?
- Which identifiers must be **unique**, and unique where?

---
## 2. Write and save your JSON

Each cell below writes one file in `sources/`. Replace `{}` with your content, following **your** tables.

Tips
- every record that others must refer to needs an `id`;
- refer to pieces and colours **by their id**, never by a description;
- use the **shared catalogue** ids, so that your model can be merged with the other groups' models;
- JSON wants double quotes `"…"`, no trailing commas, no comments.

### 2a. The catalogue (pieces and colours)
If the teacher gives you a shared `catalogue.json`, upload it to `sources/` and skip this cell.

In [ ]:
%%writefile sources/catalogue.json
{}

### 2b. Your model

In [ ]:
%%writefile sources/model.json
{}

### 2c. Load the files
If a file has a syntax error, `json.load` tells you the line and column: fix the cell above and run it again.

In [ ]:
with open("sources/catalogue.json") as f:
    catalogue = json.load(f)
with open("sources/model.json") as f:
    model = json.load(f)

print(json.dumps(model, indent=2)[:1000])

---
## 3. Check your data

Write your own checks with plain Python loops. At least:
1. every piece and colour used in the model **exists in the catalogue**;
2. the total number of pieces declared in the model **equals the sum of the quantities** in the layers below.

*Optional:* write your tables as a JSON Schema and validate with `from jsonschema import validate`.

In [ ]:
# your checks here

---
## 4. Build the graph with networkx

You decide what becomes a **node** and what becomes an **edge**. Some building blocks:

```python
G = nx.DiGraph()        # directed graph: edges have a direction (A → B)
G = nx.MultiDiGraph()   # same, but allows several edges between the same two nodes

G.add_node("brick-2x4", label="Brick 2x4", layer=2)           # a node, with any attributes you like
G.add_edge("house-walls", "brick-2x4", quantity=6, colour="white")   # an edge, with attributes

G.nodes(data=True)        # all nodes with their attributes
G.edges(data=True)        # all edges with their attributes
G.predecessors("brick-2x4")   # nodes with an edge pointing to this node
G.number_of_nodes(), G.number_of_edges()
```

Adding a node that already exists does **not** create a copy: it is the same node.
This is how two sub-assemblies (or two models) end up sharing a piece.

Walk through your JSON with nested `for` loops, one per layer of the onion, and add nodes and edges as you go.

In [ ]:
# build your graph here

Answer with your graph (loops over `G.edges(data=True)` and `G.nodes(data=True)`):
1. **Bill of materials**: how many pieces of each kind and colour do you need?
2. Which pieces are used by **more than one** sub-assembly? In which colours?

In [ ]:
# your answers here

---
## 5. Draw the graph

Drawing needs **positions** for the nodes, then nodes, edges and labels:

```python
pos = nx.spring_layout(G, seed=1)            # automatic positions
pos = {"small-house": (0, 0), "house-walls": (1, 1), ...}   # or your own (x, y), e.g. one column per layer

fig, ax = plt.subplots(figsize=(12, 7))
nx.draw_networkx_nodes(G, pos, nodelist=[...], node_color="#1E57A8", node_shape="s", node_size=2500, ax=ax)
nx.draw_networkx_edges(G, pos, edge_color="grey", arrows=True, node_size=2500, ax=ax)
nx.draw_networkx_labels(G, pos, labels={n: d["label"] for n, d in G.nodes(data=True)}, ax=ax)
nx.draw_networkx_edge_labels(G, pos, edge_labels={("house-walls", "brick-2x4"): "6× white"}, ax=ax)
ax.axis("off")
plt.show()
```

Ideas: a different shape or colour for each layer (`node_shape`: `"o"`, `"s"`, `"h"`…);
edges coloured like the bricks; a column per layer, so that the onion is visible.

In [ ]:
# draw your graph here

---
## 6. Swap and merge

Get the `model.json` of another group, upload it to `sources/` with a different name, and add it to **your** graph.

In [ ]:
# load the other group's model and add it to your graph here

Answer:
1. Could you rebuild their model from the description alone?
2. Does their declared total match the sum of their quantities?
3. How many pieces of each kind do **the two models together** need?
4. Did any identifiers clash, or fail to match, when you merged the two models? Why?